# Temperature-Scaled Reasoning Exploration — Single Model (SmolLM-360M) on MATH-500

No teacher model here — just `HuggingFaceTB/SmolLM-360M` on its own. Instead of decoding a single
greedy solution per problem, we **sample multiple solutions at temperature > 0** for each problem
and see how much that exploration is worth, using two standard metrics:

- **pass@k**: was *at least one* of the k sampled solutions correct? (an oracle upper bound —
  useful to see how much headroom sampling exposes, even though you can't know in advance which
  sample is right)
- **self-consistency**: take a majority vote over the k sampled final answers and use that as the
  prediction. This is a practical way to actually use exploration to improve accuracy at
  inference time, without any training.

We sweep over a few temperatures so you can see how the amount of exploration (temperature +
number of samples) trades off against decoding cost and accuracy, all against a greedy baseline.

Runs fine on CPU (360M is small) but a GPU will make the sampling sweep much faster.


## Setup

In [ ]:
!pip install -q -U transformers datasets sympy tqdm


In [ ]:
import os
import re
import random
from collections import Counter

import torch
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Config

In [ ]:
model_name = "HuggingFaceTB/SmolLM-360M"

NUM_PROBLEMS = 100            # first N problems from MATH-500, per the task
MAX_NEW_TOKENS_SOLUTION = 400

NUM_SAMPLES_PER_PROBLEM = 5    # k in pass@k / self-consistency
TOP_P = 0.95
TEMPERATURES = [0.3, 0.7, 1.0]  # sweep — greedy (temperature-independent) is measured separately


## Load Dataset

In [ ]:
from datasets import load_dataset

print("Loading HuggingFaceH4/MATH-500 dataset...")
dataset = load_dataset("HuggingFaceH4/MATH-500")

# MATH-500 ships a single 'test' split; use the first NUM_PROBLEMS problems.
math_problems = dataset["test"].select(range(NUM_PROBLEMS))

print(f"Loaded {len(math_problems)} problems.")
print("Example:")
print(math_problems[0])


## Answer Extraction & Grading

MATH-500 answers are LaTeX expressions, so we pull the contents of the last `\boxed{...}` in the
generated text (falling back to the last `$...$` or number if no box is present), normalize
whitespace/formatting, and compare. When SymPy is available we also try a symbolic equivalence
check so e.g. `1/2` and `\frac{1}{2}` are treated as equal.

In [ ]:
def extract_boxed(text: str):
    """Return the contents of the last \\boxed{...} in text, handling nested braces."""
    key = "\\boxed{"
    start = text.rfind(key)
    if start == -1:
        return None
    i = start + len(key)
    depth = 1
    out = []
    while i < len(text) and depth > 0:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                break
        out.append(c)
        i += 1
    return "".join(out).strip()


def extract_predicted_answer(generated_text: str):
    boxed = extract_boxed(generated_text)
    if boxed:
        return boxed
    dollar_matches = re.findall(r"\$(.+?)\$", generated_text)
    if dollar_matches:
        return dollar_matches[-1].strip()
    num_matches = re.findall(r"-?\d+\.?\d*", generated_text)
    if num_matches:
        return num_matches[-1]
    return None


def normalize_answer(ans: str):
    if ans is None:
        return None
    a = ans.strip()
    a = a.replace("\\!", "").replace("\\,", "").replace(" ", "")
    a = a.replace("\\left", "").replace("\\right", "")
    a = a.strip("$")
    if a.endswith("."):
        a = a[:-1]
    return a


try:
    import sympy
    from sympy.parsing.latex import parse_latex

    def _sympy_equal(a: str, b: str):
        try:
            return bool(sympy.simplify(parse_latex(a) - parse_latex(b)) == 0)
        except Exception:
            return None
except Exception:
    def _sympy_equal(a: str, b: str):
        return None


def answers_match(predicted: str, true: str) -> bool:
    p, t = normalize_answer(predicted), normalize_answer(true)
    if p is None or t is None:
        return False
    if p == t:
        return True
    try:
        if float(p) == float(t):
            return True
    except (ValueError, TypeError):
        pass
    sym_result = _sympy_equal(p, t)
    if sym_result is not None:
        return sym_result
    return False


## Load Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()
print("Model loaded.")


## Generation Helper (greedy or temperature-sampled, single or multi-sample)

In [ ]:
SOLUTION_INSTRUCTION = (
    "Solve the following math problem step by step. "
    "End your response with the final answer written as \\boxed{{answer}}.\n\n"
    "Problem: {problem}"
)


def build_prompt(tokenizer, problem: str) -> str:
    user_msg = SOLUTION_INSTRUCTION.format(problem=problem)
    if getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": user_msg}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # SmolLM-360M is a base (non-instruct) model — plain-text prompt.
    return f"{user_msg}\nSolution:"


@torch.no_grad()
def generate(model, tokenizer, problem: str, max_new_tokens=MAX_NEW_TOKENS_SOLUTION,
             do_sample=False, temperature=1.0, top_p=0.95, num_return_sequences=1):
    """Generate one or more completions. Greedy when do_sample=False (temperature ignored);
    temperature-scaled sampling when do_sample=True — this is the 'reasoning exploration' mode."""
    prompt = build_prompt(tokenizer, problem)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        num_return_sequences=num_return_sequences,
    )
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        gen_kwargs.update(do_sample=False)

    outputs = model.generate(**inputs, **gen_kwargs)
    prompt_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(o[prompt_len:], skip_special_tokens=True) for o in outputs]


## Greedy Baseline

In [ ]:
correct = 0
for problem in tqdm(math_problems, desc="Greedy baseline"):
    generated_text = generate(model, tokenizer, problem["problem"], do_sample=False)[0]
    predicted = extract_predicted_answer(generated_text)
    if answers_match(predicted, problem["answer"]):
        correct += 1

greedy_accuracy = 100.0 * correct / len(math_problems)
print(f"Greedy baseline: {correct}/{len(math_problems)} correct ({greedy_accuracy:.2f}%)")


## Look at One Problem's Diversity Before Running the Full Sweep

Sanity-check what temperature sampling actually changes: compare the single greedy solution
against a handful of sampled ones for the same problem.

In [ ]:
example_problem = math_problems[0]["problem"]
example_temperature = TEMPERATURES[len(TEMPERATURES) // 2]

print("=== Greedy ===")
print(generate(model, tokenizer, example_problem, do_sample=False)[0][:500])

print(f"\n=== Sampled at temperature = {example_temperature} ===")
example_samples = generate(
    model, tokenizer, example_problem,
    do_sample=True, temperature=example_temperature, top_p=TOP_P,
    num_return_sequences=NUM_SAMPLES_PER_PROBLEM,
)
for i, s in enumerate(example_samples):
    print(f"\n--- sample {i} ---")
    print(s[:350])


## Temperature Sweep: pass@k and Self-Consistency

For each temperature, sample `NUM_SAMPLES_PER_PROBLEM` solutions per problem and compute:

- **pass@k** — correct if any sample's answer matches the true answer (oracle, not achievable
  without knowing the ground truth, but shows how much correct reasoning is *in there somewhere*).
- **self-consistency** — group the k sampled final answers, take the majority, and check whether
  that majority answer is correct. This is a usable inference-time strategy.

In [ ]:
def self_consistency_prediction(raw_predictions):
    """Group by normalized answer string and return a representative of the majority group."""
    valid = [p for p in raw_predictions if p is not None]
    if not valid:
        return None
    normalized = [normalize_answer(p) for p in valid]
    counts = Counter(normalized)
    majority_norm, _ = counts.most_common(1)[0]
    # Return the first raw prediction whose normalized form matches the majority group.
    for raw, norm in zip(valid, normalized):
        if norm == majority_norm:
            return raw
    return None


sweep_results = {}  # temperature -> {"pass_at_k": %, "self_consistency": %}

for temperature in TEMPERATURES:
    pass_at_k_correct = 0
    self_consistency_correct = 0

    for problem in tqdm(math_problems, desc=f"Sampling @ T={temperature}"):
        samples = generate(
            model, tokenizer, problem["problem"],
            do_sample=True, temperature=temperature, top_p=TOP_P,
            num_return_sequences=NUM_SAMPLES_PER_PROBLEM,
        )
        raw_predictions = [extract_predicted_answer(s) for s in samples]

        if any(answers_match(p, problem["answer"]) for p in raw_predictions):
            pass_at_k_correct += 1

        majority_pred = self_consistency_prediction(raw_predictions)
        if answers_match(majority_pred, problem["answer"]):
            self_consistency_correct += 1

    n = len(math_problems)
    sweep_results[temperature] = {
        "pass_at_k": 100.0 * pass_at_k_correct / n,
        "self_consistency": 100.0 * self_consistency_correct / n,
    }
    print(f"T={temperature}: pass@{NUM_SAMPLES_PER_PROBLEM} = {sweep_results[temperature]['pass_at_k']:.2f}%, "
          f"self-consistency = {sweep_results[temperature]['self_consistency']:.2f}%")


## Results Summary

In [ ]:
print(f"{'Method':<30}{'Accuracy':>10}")
print("-" * 40)
print(f"{'Greedy (T=0)':<30}{greedy_accuracy:>9.2f}%")
for temperature, res in sweep_results.items():
    print(f"{'pass@' + str(NUM_SAMPLES_PER_PROBLEM) + f' (T={temperature})':<30}{res['pass_at_k']:>9.2f}%")
    print(f"{'self-consistency (T=' + str(temperature) + ')':<30}{res['self_consistency']:>9.2f}%")


In [ ]:
import matplotlib.pyplot as plt

temps = list(sweep_results.keys())
pass_at_k_vals = [sweep_results[t]["pass_at_k"] for t in temps]
self_consistency_vals = [sweep_results[t]["self_consistency"] for t in temps]

fig, ax = plt.subplots(figsize=(7, 5))
ax.axhline(greedy_accuracy, color="gray", linestyle="--", label="Greedy baseline")
ax.plot(temps, pass_at_k_vals, marker="o", label=f"pass@{NUM_SAMPLES_PER_PROBLEM} (oracle)")
ax.plot(temps, self_consistency_vals, marker="s", label="self-consistency (majority vote)")
ax.set_xlabel("Sampling temperature")
ax.set_ylabel("Accuracy on MATH-500 (%)")
ax.set_title(f"SmolLM-360M: reasoning exploration vs greedy (k={NUM_SAMPLES_PER_PROBLEM})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Notes & Limitations

- **pass@k is an oracle upper bound**, not something you get for free — you'd need a verifier (or
  the ground-truth answer) to know *which* sample was correct. Self-consistency is the practical
  version: no ground truth needed at inference time, just a vote among samples.
- **Self-consistency here uses exact-string-after-normalization grouping**, so two samples with
  the same value written differently (`1/2` vs `\frac{1}{2}`) may end up in different vote buckets
  and split the majority. A more robust version would cluster samples using `answers_match`
  pairwise (union-find) rather than grouping by normalized string equality.
- **Cost scales with `NUM_SAMPLES_PER_PROBLEM` x `len(TEMPERATURES)` x `NUM_PROBLEMS`.** This sweep
  runs `1 + len(TEMPERATURES) x NUM_SAMPLES_PER_PROBLEM` generations per problem; increase
  `NUM_PROBLEMS`/`NUM_SAMPLES_PER_PROBLEM` gradually and watch runtime.
- **Grading is approximate** — the SymPy-based checker above catches common equivalences (simple
  fractions, algebraic rearrangements) but not everything. For rigorous MATH-style grading,
  consider the `math-verify` or `minerva_math` evaluators used in published benchmarks.
- SmolLM-360M is small and was not instruction-tuned for step-by-step math, so absolute accuracy
  numbers will likely be low across the board — the point of this notebook is the *relative*
  comparison between greedy, pass@k, and self-consistency, not chasing a high absolute score.
